## Recommendation System

#### Data Preprocessing:

In [1]:
import pandas as pd

df = pd.read_csv("anime.csv")

print(df.head())

   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type episodes  rating  \
0               Drama, Romance, School, Supernatural  Movie        1    9.37   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV       64    9.26   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.25   
3                                   Sci-Fi, Thriller     TV       24    9.17   
4  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.16   

   members  
0   200630  
1   793665  
2   114262  
3   673572  
4   151266  


In [2]:
print(df.isnull().sum())
df = df.dropna()
print(df.isnull().sum())

anime_id      0
name          0
genre        62
type         25
episodes      0
rating      230
members       0
dtype: int64
anime_id    0
name        0
genre       0
type        0
episodes    0
rating      0
members     0
dtype: int64


In [3]:
print(df.info())

print("Shape of the dataset:", df.shape)

print("Columns:")
print(df.columns)

print(df.describe())

print("\nData Types:")
print(df.dtypes)

<class 'pandas.core.frame.DataFrame'>
Index: 12017 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   anime_id  12017 non-null  int64  
 1   name      12017 non-null  object 
 2   genre     12017 non-null  object 
 3   type      12017 non-null  object 
 4   episodes  12017 non-null  object 
 5   rating    12017 non-null  float64
 6   members   12017 non-null  int64  
dtypes: float64(1), int64(2), object(4)
memory usage: 751.1+ KB
None
Shape of the dataset: (12017, 7)
Columns:
Index(['anime_id', 'name', 'genre', 'type', 'episodes', 'rating', 'members'], dtype='object')
           anime_id        rating       members
count  12017.000000  12017.000000  1.201700e+04
mean   13638.001165      6.478264  1.834888e+04
std    11231.076675      1.023857  5.537250e+04
min        1.000000      1.670000  1.200000e+01
25%     3391.000000      5.890000  2.250000e+02
50%     9959.000000      6.570000  1.552000e+03
75%  

#### Feature Extraction:

In [4]:
features = df[['genre', 'rating']]

print(features.head())

                                               genre  rating
0               Drama, Romance, School, Supernatural    9.37
1  Action, Adventure, Drama, Fantasy, Magic, Mili...    9.26
2  Action, Comedy, Historical, Parody, Samurai, S...    9.25
3                                   Sci-Fi, Thriller    9.17
4  Action, Comedy, Historical, Parody, Samurai, S...    9.16


In [5]:
features = pd.get_dummies(features, columns=['genre'])

print(features.head())

   rating  genre_Action  genre_Action, Adventure  \
0    9.37         False                    False   
1    9.26         False                    False   
2    9.25         False                    False   
3    9.17         False                    False   
4    9.16         False                    False   

   genre_Action, Adventure, Cars, Comedy, Sci-Fi, Shounen  \
0                                              False        
1                                              False        
2                                              False        
3                                              False        
4                                              False        

   genre_Action, Adventure, Cars, Mecha, Sci-Fi, Shounen, Sports  \
0                                              False               
1                                              False               
2                                              False               
3                                              F

In [6]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

features_scaled = pd.DataFrame(
    scaler.fit_transform(features),
    columns=features.columns
)
print(features_scaled.head())

     rating  genre_Action  genre_Action, Adventure  \
0  0.924370           0.0                      0.0   
1  0.911164           0.0                      0.0   
2  0.909964           0.0                      0.0   
3  0.900360           0.0                      0.0   
4  0.899160           0.0                      0.0   

   genre_Action, Adventure, Cars, Comedy, Sci-Fi, Shounen  \
0                                                0.0        
1                                                0.0        
2                                                0.0        
3                                                0.0        
4                                                0.0        

   genre_Action, Adventure, Cars, Mecha, Sci-Fi, Shounen, Sports  \
0                                                0.0               
1                                                0.0               
2                                                0.0               
3                                   

#### Recommendation System:

In [7]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(features_scaled)

def recommend_anime(anime_name):
    index = df[df["name"] == anime_name].index[0]

    similarity_scores = list(enumerate(cosine_sim[index]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    recommendations = similarity_scores[1:6]

    for i in recommendations:
        print(df.iloc[i[0]]["name"])

In [8]:
anime_name = "Naruto"

print("Recommended Anime:")
recommend_anime(anime_name)

Recommended Anime:
Naruto: Shippuuden
Boruto: Naruto the Movie - Naruto ga Hokage ni Natta Hi
Boruto: Naruto the Movie
Naruto x UT
Naruto: Shippuuden Movie 4 - The Lost Tower


In [9]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(features_scaled)

def recommend_anime(anime_name):
    index = df[df["name"] == anime_name].index[0]

    similarity_scores = list(enumerate(cosine_sim[index]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    recommended_anime = []
    for i in similarity_scores[1:6]:
        recommended_anime.append(df.iloc[i[0]]["name"])

    return recommended_anime

anime_name = "Naruto"
print("Recommended Anime:")
print(recommend_anime(anime_name))

Recommended Anime:
['Naruto: Shippuuden', 'Boruto: Naruto the Movie - Naruto ga Hokage ni Natta Hi', 'Boruto: Naruto the Movie', 'Naruto x UT', 'Naruto: Shippuuden Movie 4 - The Lost Tower']


In [10]:
def recommend_anime(anime_name, threshold=0.5):
    index = df[df["name"] == anime_name].index[0]

    similarity_scores = list(enumerate(cosine_sim[index]))

    recommendations = []

    for i, score in similarity_scores:
        if score >= threshold and i != index:
            recommendations.append(df.iloc[i]["name"])

    return recommendations

print("Threshold = 0.9")
print(recommend_anime("Naruto", threshold=0.9))

print("\nThreshold = 0.7")
print(recommend_anime("Naruto", threshold=0.7))

print("\nThreshold = 0.5")
print(recommend_anime("Naruto", threshold=0.5))

Threshold = 0.9
['Boruto: Naruto the Movie', 'Naruto: Shippuuden', 'Boruto: Naruto the Movie - Naruto ga Hokage ni Natta Hi', 'Naruto x UT', 'Naruto: Shippuuden Movie 4 - The Lost Tower', 'Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsugu Mono', 'Naruto Shippuuden: Sunny Side Battle', 'Naruto Soyokazeden Movie: Naruto to Mashin to Mitsu no Onegai Dattebayo!!']

Threshold = 0.7
['Boruto: Naruto the Movie', 'Naruto: Shippuuden', 'Boruto: Naruto the Movie - Naruto ga Hokage ni Natta Hi', 'Naruto x UT', 'Naruto: Shippuuden Movie 4 - The Lost Tower', 'Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsugu Mono', 'Naruto Shippuuden: Sunny Side Battle', 'Naruto Soyokazeden Movie: Naruto to Mashin to Mitsu no Onegai Dattebayo!!']

Threshold = 0.5
['Boruto: Naruto the Movie', 'Naruto: Shippuuden', 'Boruto: Naruto the Movie - Naruto ga Hokage ni Natta Hi', 'Naruto x UT', 'Naruto: Shippuuden Movie 4 - The Lost Tower', 'Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsugu Mono', 'Naruto Shippuuden: Sunny Sid

In [11]:
print("Number of Anime:", len(df))
print("Similarity Matrix Shape:", cosine_sim.shape)

anime_name = "Naruto"
recommendations = recommend_anime(anime_name)

print("\nRecommendations for", anime_name)
print(recommendations)

print("\nNumber of Recommendations:", len(recommendations))

Number of Anime: 12017
Similarity Matrix Shape: (12017, 12017)

Recommendations for Naruto
['Boruto: Naruto the Movie', 'Naruto: Shippuuden', 'Boruto: Naruto the Movie - Naruto ga Hokage ni Natta Hi', 'Naruto x UT', 'Naruto: Shippuuden Movie 4 - The Lost Tower', 'Naruto: Shippuuden Movie 3 - Hi no Ishi wo Tsugu Mono', 'Naruto Shippuuden: Sunny Side Battle', 'Naruto Soyokazeden Movie: Naruto to Mashin to Mitsu no Onegai Dattebayo!!']

Number of Recommendations: 8


#### Interview Questions:

In [ ]:
1. Can you explain the difference between user-based and item-based collaborative filtering?

User-Based Collaborative Filtering

Recommends items based on similar users.
Finds users with similar preferences.
Similarity is calculated between users.
Performance decreases with a large number of users.
Example: If User A and User B like similar anime, anime liked by User B are recommended to User A.

Item-Based Collaborative Filtering

Recommends items based on similar items.
Finds items that receive similar ratings from users.
Similarity is calculated between items.
More scalable and commonly used in modern recommendation systems.
Example: If users who like Naruto also like Bleach, then Bleach is recommended to someone who likes Naruto.

In [ ]:
2. What is collaborative filtering, and how does it work? solve and give

Collaborative filtering is a recommendation technique that suggests items to users based on the preferences and behavior of other users. 
It assumes that users with similar interests are likely to prefer similar items.

How it works:

Collect user-item interactions such as ratings or purchases.
Calculate similarity between users (user-based) or items (item-based).
Identify the most similar users or items.
Recommend items that similar users liked or items similar to those the user has already liked.